# 🧰 Coding agent — the `pi-coding-agent` SDK

Notebook **110** built a coding agent by hand-defining `write_file` / `run_python`
tools on the low-level `Agent`. This notebook does the same job with the **higher-level
coding-harness SDK**, `@earendil-works/pi-coding-agent`: `createAgentSession()` gives you
**built-in tools** (`read`, `write`, `bash`, `ls`, …) and a managed session, so you
mostly just pick a model, a working directory, and which tools to allow.

We reuse the `AZURE_PI_TEST_DEPLOYMENT2` deployment and confine the agent to the
`environment1/` workspace via the session's `cwd`.

> **Kernel:** Deno. The SDK is npm/Node-oriented but imports and runs under Deno; its
> `bash` tool needs run permission (the Deno Jupyter kernel provides it).

## 1. Load your `.env`

In [ ]:
import { loadEnvUp } from "playground/env";
const env = await loadEnvUp();

## 2. Register the coding deployment

Same as notebook 110: point `registerAzure()` at `AZURE_PI_TEST_DEPLOYMENT2`. This
gives us a configured `models` collection whose `streamSimple` already knows how to
talk to Azure — we'll hand that to the SDK below.

In [ ]:
import { registerAzure } from "playground/azure";

const deployment2 = Deno.env.get("AZURE_PI_TEST_DEPLOYMENT2");
const azureApiKey = Deno.env.get("AZURE_PI_TEST_API_KEY");
if (!deployment2) {
  throw new Error("Set AZURE_PI_TEST_DEPLOYMENT2 in your .env — this notebook uses it.");
}

const { models, model } = registerAzure({ deployments: [deployment2] });
console.log(`Coding agent model: azure-openai/${deployment2}`);

## 3. The workspace

The SDK's built-in tools operate relative to the session's `cwd`, so we point it at
`environment1/`. (This is a softer boundary than notebook 110's hard path check — the
`bash` tool could still `cd` elsewhere — but it keeps normal file work inside the
workspace.)

In [ ]:
import { resolve } from "node:path";

const SANDBOX = resolve(Deno.cwd(), "environment1");
await Deno.mkdir(SANDBOX, { recursive: true });
console.log(`🗂️  Workspace (session cwd): ${SANDBOX}`);

## 4. Wire Azure into the SDK

The SDK resolves a model's provider through a `ModelRegistry`. Our Azure setup is a
**custom** provider, so we register it with a `streamSimple` that delegates straight to
our `models` collection (which already carries the endpoint, auth and headers). We use
in-memory auth/registry so nothing is written to `~/.pi`.

In [ ]:
import { AuthStorage, ModelRegistry, createAgentSession } from "@earendil-works/pi-coding-agent";

const authStorage = AuthStorage.inMemory();
const modelRegistry = ModelRegistry.inMemory(authStorage);

// The key wiring: route the "azure-openai" provider through our configured collection.
// `api` is required whenever you supply `streamSimple`.
modelRegistry.registerProvider("azure-openai", {
  api: "openai-completions",
  apiKey: azureApiKey,
  streamSimple: (m, ctx, opts) => models.streamSimple(m, ctx, opts),
});

const { session } = await createAgentSession({
  cwd: SANDBOX,
  model,
  modelRegistry,
  authStorage,
  tools: ["write", "bash", "read", "ls"], // built-in tools, allowlisted
});

console.log("Session ready. Enabled tools:", session.getActiveToolNames().join(", "));

## 5. Observe the session

`session.subscribe()` streams the same lifecycle events as the low-level `Agent`
(a subset of `AgentEvent`). We print the assistant's text live and log each tool
result.

In [ ]:
const enc = new TextEncoder();
session.subscribe((event) => {
  switch (event.type) {
    case "message_update":
      if (event.assistantMessageEvent.type === "text_delta") {
        Deno.stdout.writeSync(enc.encode(event.assistantMessageEvent.delta));
      }
      break;
    case "turn_end":
      for (const tr of event.toolResults) {
        const text = tr.content.map((c) => (c.type === "text" ? c.text : "")).join("");
        console.log(`\n🔧 ${tr.toolName}:\n${text.slice(0, 600)}`);
      }
      break;
    case "agent_end":
      console.log("\n\n✅ Session run finished.");
      break;
  }
});
console.log("Subscribed to session events.");

## 6. Run it — write & execute Python

`session.prompt()` runs the built-in agent loop: the model uses `write` to create the
file and `bash` to run it, then reports the output.

In [ ]:
await session.prompt(
  "Create a file `hello.py` in the current directory that prints exactly: Hello, World! " +
  "Then run it with `python3 hello.py` and tell me the output.",
);
await session.agent.waitForIdle();

## 7. Proof: show and re-run the generated file

In [ ]:
const helloPath = resolve(SANDBOX, "hello.py");
console.log("📄 environment1/hello.py:\n");
console.log(await Deno.readTextFile(helloPath));

const proof = await new Deno.Command("python3", {
  args: [helloPath],
  cwd: SANDBOX,
  stdout: "piped",
  stderr: "piped",
}).output();
console.log("\n▶️  python3 hello.py →");
console.log(new TextDecoder().decode(proof.stdout));

## ✅ What just happened — and 110 vs 120

`createAgentSession()` gave us a managed coding agent with **built-in tools**; we only
had to (1) register our custom Azure provider's `streamSimple` on a `ModelRegistry`,
(2) pick `cwd` + a tool allowlist, and (3) `prompt()`.

| | **110 — pi-agent-core** | **120 — pi-coding-agent (this)** |
|---|---|---|
| Tools | you write `execute` yourself | built-in `read`/`write`/`bash`/`ls`/… |
| Sandbox | hard per-path check (`..` refused) | soft: session `cwd` |
| Wiring | `streamFn` on `Agent` | `registerProvider({ streamSimple })` + `ModelRegistry` |
| Best for | precise, custom/safe tools | quick, batteries-included coding agents |
| Extends via | more `AgentTool`s | extensions & custom tools |

Both drive the same Azure deployment and the same underlying loop — the SDK just trades
control for convenience.

🎉 **Series complete.** From `000` (TypeScript runs) → `010` (one round-trip) →
streaming, chat, tools, structured output, vision, reasoning, cost/robustness,
multi-provider, the agent framework, and finally two flavors of a real coding agent.